# Logit Lens — Memory vs. Context Conflict Probe

Layer-by-layer probability probe for context (doc) vs. parametric memory conflict in API tool synthesis.

For each mutation pair `(original, mutated)`, runs a forward pass with `output_hidden_states=True`,
projects each layer's residual stream through the unembedding matrix W_U, and plots
P(original) vs P(mutated) at every layer depth to find where the model commits.

**Now supports all 14 datasets with automatic path resolution and clean-pair filtering.**
Pairs where `original` and `mutated` share the same first BPE token are automatically skipped,
since their logit lens curves would be trivially identical up to the disambiguation point.

In [ ]:
# ── Install / upgrade deps (Krylov base env may be missing these) ─────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--user",
     "jinja2>=3.1.0", "transformers>=4.40,<5.0", "accelerate"],
    check=True,
)

import torch, transformers, matplotlib, jinja2
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"jinja2       {jinja2.__version__}")
print(f"matplotlib   {matplotlib.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import os
# ── Environment — set before any HF imports ───────────────────────────────────
os.environ["HF_HOME"]      = "/home/tstereciu/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"          # skip network if model already cached
os.environ["HTTPS_PROXY"]  = "http://httpproxy-tcop.vip.ebay.com:80"
os.environ["HTTP_PROXY"]   = "http://httpproxy-tcop.vip.ebay.com:80"
# os.environ["HF_TOKEN"]   = "hf_..."       # uncomment if downloading a gated model

In [ ]:
# ── Config — edit these ───────────────────────────────────────────────────────
MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"   # or 14B / 72B if you have the VRAM
DATASET    = "zulip"                       # any of the 14 datasets listed in DATASET_CONFIGS below
PROBE_AT   = "pre"                        # "pre" = token just before param name; "at" = first token
MAX_TOKENS = 400
WINDOW     = 512   # tokens fed into hidden-states pass (centred on probe position)

# Repo root — adjust if running from a different working directory
REPO_ROOT = "/home/tstereciu/agent-tooling"   # or set to None to auto-detect via git

# Override: point to a specific mutated doc file (leave None for auto-resolve from DATASET)
DOC_PATH       = None   # e.g. "/path/to/docs_mutated/api_get-messages.md"
MUTATIONS_PATH = None   # e.g. "/path/to/mutations.json"  (None = auto from DATASET)

# ── Dataset configs ───────────────────────────────────────────────────────────
# endpoint_header: the markdown header that anchors the doc snippet to probe
# doc_file: which file in docs_mutated/ contains that endpoint (None = concatenate all)
DATASET_CONFIGS = {
    "stripe": {
        "endpoint_header": "# Create a PaymentIntent",
        "doc_file": "payment_intents.md",
    },
    "zulip": {
        "endpoint_header": "# Get messages\n",
        "doc_file": "api_get-messages.md",
    },
    "alphavantage": {
        "endpoint_header": "#### TIME_SERIES_DAILY\n",
        "doc_file": "api_time_series_data.md",
    },
    "jira": {
        "endpoint_header": "## Search for issues using JQL enhanced search (GET)",
        "doc_file": "api_issue-search.md",
    },
    "ebay_sell": {
        "endpoint_header": "# getFulfillmentPolicies",
        "doc_file": None,   # concat all
    },
    "notion": {
        # query-a-data-source.md contains query_filter, sort_rules, results_per_page,
        # page_cursor, is_archived — all 5 mutated params in one doc
        "endpoint_header": "# query-a-data-source",
        "doc_file": "query-a-data-source.md",
    },
    "github": {
        # verified: exists at line 762 of api_issues-issues.md
        "endpoint_header": "## List repository issues",
        "doc_file": "api_issues-issues.md",
    },
    "ebay_buy": {
        # no endpoint-name header; file has ## Input / ## Output sections only
        # use ## Input which immediately precedes the parameter table
        "endpoint_header": "## Input",
        "doc_file": "api_buy_browse_resources_item_summary_methods_search.md",
    },
    "ebay_commerce": {
        # same structure as ebay_buy
        "endpoint_header": "## Input",
        "doc_file": "api_commerce_catalog_resources_product_summary_methods_search.md",
    },
    "confluence": {
        # verified: exists at line 25 of api_v1_search.md
        "endpoint_header": "## Search content",
        "doc_file": "api_v1_search.md",
    },
    "shopify": {
        # Shopify REST docs embed query params inside response examples with no clean header.
        # "## The Order resource" gives the resource field descriptions (payment_status,
        # shipping_status etc.) which is enough context for the model to use them as params.
        "endpoint_header": "## The Order resource",
        "doc_file": "api_order.md",
    },
    "mastodon": {
        # verified: actual header is "## Post a new status" in api_statuses.md
        "endpoint_header": "## Post a new status",
        "doc_file": "api_statuses.md",
    },
    "openweathermap": {
        # H1 level header at top of file; extract_doc_section will use the full file
        "endpoint_header": "# Current Weather Data (API 2.5)",
        "doc_file": "api_current_weather.md",
    },
    "spoonacular": {
        # verified: exists at line 7 of api_recipes.md
        "endpoint_header": "## Search Recipes",
        "doc_file": "api_recipes.md",
    },
}

cfg = DATASET_CONFIGS[DATASET]
print(f'Dataset: {DATASET}  Model: {MODEL_ID}  Probe: {PROBE_AT}')
print(f'Header:  {cfg["endpoint_header"]!r}')

In [ ]:
# ── Auto-resolve doc and mutations paths from repo ────────────────────────────
import subprocess, json, math, textwrap
from pathlib import Path

def find_repo_root(hint=None):
    if hint:
        return Path(hint)
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "--show-toplevel"], stderr=subprocess.DEVNULL
        )
        return Path(out.decode().strip())
    except Exception:
        return Path(".").resolve()

repo_root = find_repo_root(REPO_ROOT)
dataset_dir = repo_root / "benchmark" / "datasets" / DATASET

# ── Load mutations ─────────────────────────────────────────────────────────────
if MUTATIONS_PATH:
    mutations_all = json.loads(Path(MUTATIONS_PATH).read_text())["mutations"]
else:
    mutations_all = json.loads((dataset_dir / "mutations.json").read_text())["mutations"]

# ── Load doc (mutated version, which has the renamed params) ──────────────────
if DOC_PATH:
    raw_doc = Path(DOC_PATH).read_text()
else:
    docs_dir = dataset_dir / "docs_mutated"
    doc_file = cfg["doc_file"]
    if doc_file:
        raw_doc = (docs_dir / doc_file).read_text()
    else:
        # Concatenate all .md files in docs_mutated/
        parts = sorted(docs_dir.glob("*.md"))
        raw_doc = "\n\n".join(p.read_text() for p in parts)

# ── Extract relevant doc section ──────────────────────────────────────────────
def extract_doc_section(text: str, header: str, char_limit: int = 6000) -> str:
    start = text.rfind(header.rstrip())
    if start == -1:
        print(f"  WARNING: header not found: {header!r}  — using full doc")
        return text[:char_limit]
    stripped = header.lstrip("#")
    level = len(header) - len(stripped) - (1 if stripped.startswith(" ") else 0)
    level = max(level, 1)
    next_h = text.find("\n" + "#" * level + " ", start + len(header))
    end = next_h if next_h != -1 else len(text)
    return text[start:end].strip()[:char_limit]

doc_snippet = extract_doc_section(raw_doc, cfg["endpoint_header"])
print(f"Repo root  : {repo_root}")
print(f"Doc chars  : {len(doc_snippet)}")
print(f"Mutations  : {[m['original']+'→'+m['mutated'] for m in mutations_all]}")
print()
print(doc_snippet[:500], "...")

In [ ]:
# ── Prompt templates ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are an expert Python developer building MCP (Model Context Protocol) tool servers.
Given an API documentation snippet, write a single Python function that wraps the described endpoint.

Rules:
- Use @mcp.tool() decorator
- Function parameters must exactly match the parameter names documented (spelling matters)
- Use requests to make the HTTP call
- Pass parameters as a dict called `params` for query params or `data` for form-encoded bodies
- Return the JSON response as a dict
- Keep it concise — no docstring needed, just the function
"""

HUMAN_TEMPLATE = """\
Here is the API documentation for one endpoint:

---
{doc_snippet}
---

Write a single Python @mcp.tool() function that implements this endpoint.
Parameter names in your function signature MUST match the documentation exactly.
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": HUMAN_TEMPLATE.format(doc_snippet=doc_snippet)},
]
print('Prompt constructed.')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
import site, sys

# Include user site-packages (pip install --user puts packages here)
_user_site = site.getusersitepackages()
if _user_site not in sys.path:
    sys.path.insert(0, _user_site)

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.bfloat16 if device.type == "cuda" else torch.float32
print(f"Device: {device}  dtype: {dtype}")
if device.type == "cuda":
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.0f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,          # use dtype= (not torch_dtype=) for compatibility with older transformers
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
n_layers   = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
n_params   = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nLoaded {MODEL_ID}")
print(f"  {n_params:.1f}B params | {n_layers} layers | hidden_dim={hidden_dim}")

In [ ]:
# ── Tokenize prompt & generate response ───────────────────────────────────────
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
)
if hasattr(prompt_ids, "input_ids"):
    prompt_ids = prompt_ids.input_ids
prompt_ids = prompt_ids.to(device)
print(f"Prompt: {prompt_ids.shape[1]} tokens — generating up to {MAX_TOKENS} more...")

with torch.no_grad():
    gen_ids = model.generate(
        prompt_ids,
        max_new_tokens=MAX_TOKENS,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

response_start = prompt_ids.shape[1]
response_ids   = gen_ids[0, response_start:]
response_text  = tokenizer.decode(response_ids, skip_special_tokens=True)
full_ids       = gen_ids[0].tolist()

print(f"Response ({len(response_ids)} tokens):")
print(textwrap.indent(response_text, "  "))

In [ ]:
# ── Resolve first sub-token ID and filter clean pairs ─────────────────────────
def get_first_token_ids(tokenizer, names):
    """Return {name: first_bpe_token_id} for each name."""
    result = {}
    for name in names:
        for prefix in (" ", "", "\n    ", "\n"):
            ids = tokenizer.encode(prefix + name, add_special_tokens=False)
            if ids:
                result[name] = ids[0]
                break
    return result

def find_probe_positions(token_ids_seq, tokenizer, target_names, probe_at="pre"):
    """Find where each param name first appears in the generated token stream."""
    parts = [tokenizer.decode([t]) for t in token_ids_seq]
    cumulative, running = [], ""
    for p in parts:
        running += p
        cumulative.append(running)
    positions = {}
    for name in target_names:
        pos = None
        for i, (before, at) in enumerate(zip([""] + cumulative[:-1], cumulative)):
            if name not in before and name in at:
                pos = max(0, i - 1) if probe_at == "pre" else i
                break
        positions[name] = pos
    return positions

all_names = [n for m in mutations_all for n in (m["original"], m["mutated"])]
name_to_tid = get_first_token_ids(tokenizer, all_names)

# ── Filter to clean pairs: original and mutated must have DIFFERENT first tokens
mutations = []
skipped   = []
for m in mutations_all:
    orig_tid = name_to_tid.get(m["original"])
    mut_tid  = name_to_tid.get(m["mutated"])
    if orig_tid is not None and mut_tid is not None and orig_tid != mut_tid:
        mutations.append(m)
    else:
        skipped.append(m)

probe_positions = find_probe_positions(
    full_ids[response_start:], tokenizer, all_names, probe_at=PROBE_AT
)

print(f"Clean pairs ({len(mutations)}):")
for m in mutations:
    o, mu = m["original"], m["mutated"]
    o_tid, mu_tid = name_to_tid[o], name_to_tid[mu]
    print(f"  {o!r:25s} (tok {o_tid:6d} = {tokenizer.decode([o_tid])!r})"
          f"  →  {mu!r:25s} (tok {mu_tid:6d} = {tokenizer.decode([mu_tid])!r})")
if skipped:
    print(f"\nSkipped {len(skipped)} pairs with shared first token:")
    for m in skipped:
        o, mu = m["original"], m["mutated"]
        print(f"  {o!r} → {mu!r}  (both tok {name_to_tid.get(o)!r} = {tokenizer.decode([name_to_tid[o]])!r})"
              if name_to_tid.get(o) == name_to_tid.get(mu) else f"  {o!r} → {mu!r}  (token id missing)")

print()
print("Probe positions in response:")
for m in mutations:
    for name in (m["original"], m["mutated"]):
        pos = probe_positions.get(name)
        print(f"  {name!r:25s} → tok {pos}")

In [ ]:
# ── Logit lens helper ─────────────────────────────────────────────────────────
W_U = model.lm_head.weight.detach().float()   # (vocab_size, hidden_size)
has_ln = hasattr(model, "model") and hasattr(model.model, "norm")

def logit_lens_at_position(hidden_states, position, token_ids):
    """
    For each layer, project hidden state at `position` through W_U → softmax.
    Applies the final layer-norm to intermediate layers (pre-LN logit lens).
    Returns list of {layer, probs, logits} dicts.
    """
    results = []
    n = len(hidden_states)
    for l, hs in enumerate(hidden_states):
        h = hs[0, position, :].float().cpu()
        # Apply final LN to all but last layer (last already has LN applied before lm_head)
        if has_ln and l < n - 1:
            with torch.no_grad():
                h = model.model.norm(
                    h.to(next(model.parameters()).device).unsqueeze(0)
                ).squeeze(0).float().cpu()
        logits    = W_U.cpu() @ h
        log_probs = F.log_softmax(logits, dim=-1)
        results.append({
            "layer":  l,
            "probs":  {tid: math.exp(log_probs[tid].item()) for tid in token_ids},
            "logits": {tid: logits[tid].item()              for tid in token_ids},
        })
    return results

print("Logit lens helper ready.")

In [ ]:
# ── Run logit lens for each clean mutation pair ───────────────────────────────
all_results = []

for mutation in mutations:
    orig = mutation["original"]
    mut  = mutation["mutated"]

    orig_pos = probe_positions.get(orig)
    mut_pos  = probe_positions.get(mut)

    # Determine which name actually appeared in the generated response
    if mut_pos is not None:
        winner_name, winner_pos, outcome = mut, mut_pos, "doc"
    elif orig_pos is not None:
        winner_name, winner_pos, outcome = orig, orig_pos, "memory"
    else:
        print(f"  SKIP  {orig}→{mut}: neither name found in response")
        continue

    abs_pos   = response_start + winner_pos
    abs_pos   = min(abs_pos, len(full_ids) - 1)
    win_start = max(0, abs_pos - WINDOW + 1)
    window    = full_ids[win_start : abs_pos + 1]
    pos_in_w  = len(window) - 1

    tag = "DOC WON   " if outcome == "doc" else "MEMORY WON"
    print(f"  [{tag}]  {orig!r} ↔ {mut!r}"
          f"  (window {win_start}:{abs_pos+1}, {len(window)} tokens)", flush=True)

    win_tensor = torch.tensor([window], device=device)
    with torch.no_grad():
        out = model(win_tensor, output_hidden_states=True)
    hidden_states = out.hidden_states
    del out, win_tensor
    if device.type == "cuda":
        torch.cuda.empty_cache()

    orig_tid = name_to_tid.get(orig)
    mut_tid  = name_to_tid.get(mut)
    tids     = [t for t in [orig_tid, mut_tid] if t is not None]

    layer_data = logit_lens_at_position(hidden_states, pos_in_w, tids)
    del hidden_states

    # ── Per-layer table ──────────────────────────────────────────────────────
    col_o = f"P({orig})"
    col_m = f"P({mut})"
    print(f"  {'Layer':>10}  {col_o:>24}  {col_m:>24}  {'Leader':>14}  {'Gap':>8}")
    print(f"  {'-'*88}")
    prev_leader, flip_layer = None, None
    for ld in layer_data:
        l    = ld["layer"]
        p_o  = ld["probs"].get(orig_tid, 0.0) if orig_tid else 0.0
        p_m  = ld["probs"].get(mut_tid,  0.0) if mut_tid  else 0.0
        leader = orig if p_o >= p_m else mut
        gap    = abs(math.log(p_o + 1e-30) - math.log(p_m + 1e-30))
        bar_o  = int(p_o / max(p_o + p_m, 1e-9) * 20)
        bar    = "O" * bar_o + "M" * (20 - bar_o)
        pct    = l / max(n_layers, 1) * 100
        phase  = "early" if pct < 33 else "mid  " if pct < 66 else "late "
        label  = f"{phase} L{l:02d}"
        if prev_leader is not None and leader != prev_leader and flip_layer is None:
            flip_layer = l
            print(f"  {'':>10}  ← LEAD FLIPS at layer {l}")
        print(f"  {label:>10}  {p_o:>24.6f}  {p_m:>24.6f}  {leader:>14s}  {gap:>8.2f}  [{bar}]")
        prev_leader = leader
    print()

    all_results.append({
        "mutation":   mutation,
        "outcome":    outcome,
        "probe_pos":  abs_pos,
        "flip_layer": flip_layer,
        "layers":     layer_data,
    })

In [ ]:
# ── Plot: P(orig) vs P(mut) across layers ────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

n_mutations = len(all_results)
if n_mutations == 0:
    print("No results to plot — check that clean pairs appeared in the response.")
else:
    cols = min(n_mutations, 4)
    rows = math.ceil(n_mutations / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    ax_flat = axes.flatten()

    for ax, res in zip(ax_flat, all_results):
        orig = res["mutation"]["original"]
        mut  = res["mutation"]["mutated"]
        orig_tid = name_to_tid.get(orig)
        mut_tid  = name_to_tid.get(mut)

        layers  = [ld["layer"] for ld in res["layers"]]
        p_origs = [ld["probs"].get(orig_tid, 0.0) if orig_tid else 0.0 for ld in res["layers"]]
        p_muts  = [ld["probs"].get(mut_tid,  0.0) if mut_tid  else 0.0 for ld in res["layers"]]

        ax.plot(layers, p_origs, label=f"P({orig})\n[memory]", color="tomato",    linewidth=2)
        ax.plot(layers, p_muts,  label=f"P({mut})\n[doc]",     color="steelblue", linewidth=2)

        if res["flip_layer"] is not None:
            ax.axvline(res["flip_layer"], color="gray", linestyle="--", alpha=0.7,
                       label=f"flip @ L{res['flip_layer']}")

        outcome_label = "DOC WON" if res["outcome"] == "doc" else "MEMORY WON"
        color = "steelblue" if res["outcome"] == "doc" else "tomato"
        ax.set_title(f"{orig} → {mut}\n[{outcome_label}]", fontsize=9, color=color, fontweight="bold")
        ax.set_xlabel("Layer", fontsize=8)
        ax.set_ylabel("Probability", fontsize=8)
        ax.legend(fontsize=7)
        ax.set_xlim(0, n_layers)
        ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for ax in ax_flat[n_mutations:]:
        ax.set_visible(False)

    model_short = MODEL_ID.split("/")[-1]
    n_doc    = sum(1 for r in all_results if r["outcome"] == "doc")
    n_memory = sum(1 for r in all_results if r["outcome"] == "memory")
    fig.suptitle(
        f"Logit Lens — {DATASET} / {model_short}\n"
        f"{n_doc}/{n_mutations} doc won, {n_memory}/{n_mutations} memory won  "
        f"(skipped {len(skipped)} shared-token pairs)",
        fontsize=11,
    )
    plt.tight_layout()
    fname_png = f"logit_lens_{DATASET}.png"
    plt.savefig(fname_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname_png}")

In [ ]:
# ── Save raw results to JSON ───────────────────────────────────────────────────
import json

out = {
    "dataset":       DATASET,
    "model":         MODEL_ID,
    "probe_at":      PROBE_AT,
    "response_text": response_text,
    "clean_pairs":   len(mutations),
    "skipped_pairs": [{"original": m["original"], "mutated": m["mutated"]} for m in skipped],
    "mutations":     mutations,
    "results": [
        {
            "mutation":   r["mutation"],
            "outcome":    r["outcome"],
            "probe_pos":  r["probe_pos"],
            "flip_layer": r["flip_layer"],
            "layers": [
                {
                    "layer":  ld["layer"],
                    "probs":  {str(k): v for k, v in ld["probs"].items()},
                    "logits": {str(k): v for k, v in ld["logits"].items()},
                }
                for ld in r["layers"]
            ],
        }
        for r in all_results
    ],
}

fname_json = f"logit_lens_{DATASET}.json"
with open(fname_json, "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved {fname_json}  ({len(all_results)} pairs, {len(skipped)} skipped)")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Pair':<40}  {'Outcome':>10}  {'Flip layer':>12}  {'Final gap':>12}")
print("-" * 78)
for r in all_results:
    orig = r["mutation"]["original"]
    mut  = r["mutation"]["mutated"]
    orig_tid = name_to_tid.get(orig)
    mut_tid  = name_to_tid.get(mut)
    last = r["layers"][-1]
    p_o = last["probs"].get(orig_tid, 0.0) if orig_tid else 0.0
    p_m = last["probs"].get(mut_tid,  0.0) if mut_tid  else 0.0
    gap = abs(math.log(p_o + 1e-30) - math.log(p_m + 1e-30))
    flip = str(r["flip_layer"]) if r["flip_layer"] is not None else "never"
    pair = f"{orig} → {mut}"
    print(f"{pair:<40}  {r['outcome']:>10}  {flip:>12}  {gap:>12.2f} nats")
print()
print(f"Doc won   : {sum(1 for r in all_results if r['outcome']=='doc')}")
print(f"Memory won: {sum(1 for r in all_results if r['outcome']=='memory')}")